In [ ]:
%pip install torchinfo

# DL модели для прогнозирования временных рядов

Посмотрим на деле на те модели, которые проходили на прошлых лекциях.

__Содержание:__
0. Подготовка данных
1. DLinear — выделение тренда и остатка + линейные слои для каждой компоненты
2. CNN (TCNForecaster) + RNN (LSTNet)
3. Интервальное прогнозирование с экзогенными переменными в TFT (Supervised Transformer Model)

Бонус:
- Поговорим про фреймворки, которые можно использовать.
- Поговорим про фунд. модели.

In [ ]:
import json
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary

In [ ]:
def choose_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using GPU")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS")
    else:
        device = torch.device("cpu")
        print("Using CPU")
    return device


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()
DEVICE = choose_device()

Using GPU


In [ ]:
# Sampling parameters
START_DATE = "2016-07-01"
END_DATE = "2017-08-15"
NUM_SERIES = 100

# Validation and test split
VAL_START = pd.Timestamp("2017-06-16")
TEST_START = pd.Timestamp("2017-07-16")

# Forecasting parameters
L = 90
H = 14

# Training parameters
BATCH_SIZE = 256
NUM_WORKERS = 0

## 0. Подготовка данных

Будем работать с данными [Corporación Favorita Grocery Sales Forecasting](https://www.kaggle.com/c/favorita-grocery-sales-forecasting), которые представляют собой продажи в магазинах сети супермаркетов.

__Загрузим данные и просемплируем их, чтобы было не так тяжело обучать модели.__

In [ ]:
!mkdir data

In [ ]:
# Загружаем данные
train = pd.read_csv(f"data/train.csv", parse_dates=["date"]).drop(columns=["id"])
transactions = pd.read_csv(f"data/transactions.csv", parse_dates=["date"])
oil = pd.read_csv(f"data/oil.csv", parse_dates=["date"])
hol = pd.read_csv(f"data/holidays_events.csv", parse_dates=["date"])

# Фильтруем по датам
train = train[(train["date"] >= START_DATE) & (train["date"] <= END_DATE)]
transactions = transactions[
    (transactions["date"] >= START_DATE) & (transactions["date"] <= END_DATE)
]
oil = oil[(oil["date"] >= START_DATE) & (oil["date"] <= END_DATE)]
hol = hol[(hol["date"] >= START_DATE) & (hol["date"] <= END_DATE)]

# Отберем топ-NUM_SERIES временных рядов по сумме продаж
top = train.groupby(["store_nbr", "item_nbr"])["unit_sales"].sum().sort_values(ascending=False)
top_keys = list(top.head(NUM_SERIES).index)
train = train[train.set_index(["store_nbr", "item_nbr"]).index.isin(top_keys)]

print(train.shape, transactions.shape, oil.shape, hol.shape)
train.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

__Соберем датасет признаков.__

In [ ]:
# # Датасет праздников
# hol_day = (
#     hol[~hol["type"].isin(["Work Day"])].groupby("date").size().rename("holiday_cnt").reset_index()
# )
# hol_day["is_holiday"] = (hol_day["holiday_cnt"] > 0).astype(np.float32)
# hol_day = hol_day[["date", "is_holiday"]]

# # Собираем датасет признаков
# df = train.merge(transactions, on=["date", "store_nbr"], how="left")
# df = df.merge(oil, on="date", how="left")
# df = df.merge(hol_day, on="date", how="left")

# df["transactions"] = df["transactions"].fillna(0).astype(np.float32)
# df["is_holiday"] = df["is_holiday"].fillna(0).astype(np.float32)
# df["onpromotion"] = df["onpromotion"].fillna(0).astype(np.float32)
# df["unit_sales"] = df["unit_sales"].astype(np.float32)
# df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill().astype(np.float32)

# # Добавляем календарные признаки
# df["dow"] = df["date"].dt.dayofweek.astype(np.int64)
# df["month"] = df["date"].dt.month.astype(np.int64)

# df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7).astype(np.float32)
# df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7).astype(np.float32)
# df["mon_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype(np.float32)
# df["mon_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype(np.float32)
# df["is_weekend"] = (df["dow"] >= 5).astype(np.float32)

# # Перейдем к логарифму от таргета и некоторых признаков
# df["y"] = np.log1p(df["unit_sales"]).astype(np.float32)
# df["oil_log"] = np.log1p(df["dcoilwtico"]).astype(np.float32)
# df["tr_log"] = np.log1p(df["transactions"]).astype(np.float32)

# # Обработаем ID
# stores = sorted(df["store_nbr"].unique().tolist())
# families = sorted(df["item_nbr"].unique().tolist())

# store2i = {s:i for i,s in enumerate(stores)}
# fam2i = {f:i for i,f in enumerate(families)}

# df["store_id"] = df["store_nbr"].map(store2i).astype(np.int64)
# df["item_id"] = df["item_nbr"].map(fam2i).astype(np.int64)

# df["y"] = df["y"].bfill().astype(np.float32)

# df = df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)

# df.head()

In [ ]:
# df = df[ID_COLS + [DATE_COL] + KNOWN_FUTURE_COLS + PAST_COV_COLS + [TARGET_COL]]
# df.head()

In [ ]:
# df.to_csv("data/processed.csv", index=False)

In [ ]:
df = pd.read_csv("/content/processed.csv")
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df.head()

,store_id,item_id,date,onpromotion,is_holiday,is_weekend,dow_sin,dow_cos,mon_sin,mon_cos,tr_log,oil_log,y
0,0,2,2016-07-01,1.0,0.0,0.0,-0.433884,-0.900969,-0.5,-0.866025,7.605890,3.912423,5.539926
1,0,2,2016-07-02,0.0,0.0,1.0,-0.974928,-0.222521,-0.5,-0.866025,7.652070,3.912423,4.869371
2,0,2,2016-07-03,0.0,1.0,1.0,-0.781832,0.623490,-0.5,-0.866025,7.615298,3.912423,3.869136
3,0,2,2016-07-04,0.0,0.0,0.0,0.000000,1.000000,-0.5,-0.866025,7.531552,3.912423,4.968660
4,0,2,2016-07-05,0.0,0.0,0.0,0.781832,0.623490,-0.5,-0.866025,7.484930,3.865560,4.696125


In [ ]:
TARGET_COL = "y"
DATE_COL = "date"
ID_COLS = ["store_id", "item_id"]
KNOWN_FUTURE_COLS = [
    "onpromotion",
    "is_holiday",
    "is_weekend",
    "dow_sin",
    "dow_cos",
    "mon_sin",
    "mon_cos",
]
PAST_COV_COLS = [
    "tr_log",
    "oil_log",
]  # То есть для этих признаков мы будем использовать только прошлые значения

In [ ]:
fig = go.Figure()

for store in df["store_id"].unique()[:1]:
    for item in df[df["store_id"] == store]["item_id"].unique()[:2]:
        current_series = df[(df["store_id"] == store) & (df["item_id"] == item)].sort_values(
            "date"
        )
        for col in [TARGET_COL] + PAST_COV_COLS:
            fig.add_trace(
                go.Scatter(
                    x=current_series["date"],
                    y=current_series[col],
                    mode="lines",
                    name=f"Store {store} Item {item} - {col}",
                    line=dict(width=1),
                    opacity=0.5,
                )
            )
        for col in KNOWN_FUTURE_COLS:
            fig.add_trace(
                go.Scatter(
                    x=current_series["date"],
                    y=current_series[col],
                    mode="markers",
                    name=f"Store {store} Item {item} - {col}",
                    marker=dict(size=4),
                    opacity=0.5,
                )
            )

fig.update_layout(
    title="Временные ряды продаж и признаков для разных магазинов и товаров",
    xaxis_title="Дата",
    yaxis_title="Значение",
    legend_title="Ряды и признаки",
    height=600,
    width=1000,
    showlegend=True,
)

fig.show()

__Инициализируем Dataset и DataLoader для обучения моделей.__

In [ ]:
df

,store_id,item_id,date,onpromotion,is_holiday,is_weekend,dow_sin,dow_cos,mon_sin,mon_cos,tr_log,oil_log,y
0,0,2,2016-07-01,1.0,0.0,0.0,-0.433884,-0.900969,-0.500000,-0.866025,7.605890,3.912423,5.539926
1,0,2,2016-07-02,0.0,0.0,1.0,-0.974928,-0.222521,-0.500000,-0.866025,7.652070,3.912423,4.869371
2,0,2,2016-07-03,0.0,1.0,1.0,-0.781832,0.623490,-0.500000,-0.866025,7.615298,3.912423,3.869136
3,0,2,2016-07-04,0.0,0.0,0.0,0.000000,1.000000,-0.500000,-0.866025,7.531552,3.912423,4.968660
4,0,2,2016-07-05,0.0,0.0,0.0,0.781832,0.623490,-0.500000,-0.866025,7.484930,3.865560,4.696125
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37697,28,31,2017-08-10,0.0,1.0,0.0,0.433884,-0.900969,-0.866025,-0.500000,6.508769,3.902780,7.745003
37698,28,31,2017-08-11,0.0,1.0,0.0,-0.433884,-0.900969,-0.866025,-0.500000,6.645091,3.908216,7.204149
37699,28,31,2017-08-13,0.0,0.0,1.0,-0.781832,0.623490,-0.866025,-0.500000,6.961296,3.908216,7.538495
37700,28,31,2017-08-14,0.0,0.0,0.0,0.000000,1.000000,-0.866025,-0.500000,6.708084,3.883418,7.619234


In [ ]:
series_data = {}

for (s, f), g in df.groupby(["store_id", "item_id"], sort=False):
    g = g.sort_values("date")
    dates = g["date"].values
    y = g["y"].values.astype(np.float32)
    known = g[KNOWN_FUTURE_COLS].values.astype(np.float32)
    past_cov = g[PAST_COV_COLS].values.astype(np.float32)
    store_id = int(g["store_id"].iloc[0])
    family_id = int(g["item_id"].iloc[0])

    train_mask = g["date"] < VAL_START
    y_tr = y[train_mask.values]
    mu = float(y_tr.mean()) if len(y_tr) else 0.0
    sd = float(y_tr.std()) if len(y_tr) else 1.0
    sd = max(sd, 1e-4)

    y_n = (y - mu) / sd

    series_data[(s, f)] = dict(
        dates=dates,
        y=y_n,
        y_mu=mu,
        y_sd=sd,
        known=known,
        past_cov=past_cov,
        store_id=store_id,
        family_id=family_id,
    )

In [ ]:
print(f"Пример данных для ряда (0, 2):")
print(series_data[(0, 2)].keys())
print(series_data[(0, 2)]["y"][:5])

Пример данных для ряда (0, 2):
dict_keys(['dates', 'y', 'y_mu', 'y_sd', 'known', 'past_cov', 'store_id', 'family_id'])
[ 0.82671785  0.09939552 -0.98551804  0.2070896  -0.08851714]


In [ ]:
def build_window_index(dates, L, H, val_start, test_start):
    n = len(dates)
    idx_train, idx_val = [], []
    for t in range(L - 1, n - H):
        horizon_end = pd.Timestamp(dates[t + H])
        if horizon_end < val_start:
            idx_train.append(t)
        elif horizon_end < test_start:
            idx_val.append(t)
    t_test = n - H - 1
    if t_test >= L - 1:
        return idx_train, idx_val, [t_test]
    return idx_train, idx_val, []


class WindowDataset(Dataset):
    def __init__(self, series_data, keys, t_indices_by_key, L, H):
        self.items = []
        self.series_data = series_data
        self.L = L
        self.H = H
        for k in keys:
            for t in t_indices_by_key[k]:
                self.items.append((k, t))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        (k, t) = self.items[i]
        d = self.series_data[k]
        y = d["y"]
        known = d["known"]
        past_cov = d["past_cov"]

        x_y = y[t - self.L + 1 : t + 1]  # (L, )
        x_past_cov = past_cov[t - self.L + 1 : t + 1, :]  # (L, Cp)
        x_known_past = known[t - self.L + 1 : t + 1, :]  # (L, Cf)
        x_known_fut = known[t + 1 : t + self.H + 1, :]  # (H, Cf)

        y_fut = y[t + 1 : t + self.H + 1]  # (H, )

        return {
            "x_y": torch.tensor(x_y).float(),  # (L, )
            "x_past_cov": torch.tensor(x_past_cov).float(),  # (L, Cp)
            "x_known_past": torch.tensor(x_known_past).float(),  # (L, Cf)
            "x_known_fut": torch.tensor(x_known_fut).float(),  # (H, Cf)
            "y": torch.tensor(y_fut).float(),  # (H, )
            "store_id": torch.tensor(d["store_id"]).long(),
            "family_id": torch.tensor(d["family_id"]).long(),
        }


keys = list(series_data.keys())
tidx_train, tidx_val, tidx_test = {}, {}, {}

for k in keys:
    d = series_data[k]
    tr, va, te = build_window_index(d["dates"], L, H, VAL_START, TEST_START)
    tidx_train[k], tidx_val[k], tidx_test[k] = tr, va, te

train_ds = WindowDataset(series_data, keys, tidx_train, L, H)
val_ds = WindowDataset(series_data, keys, tidx_val, L, H)
test_ds = WindowDataset(series_data, keys, tidx_test, L, H)

print("Train dataset size:", len(train_ds))
print("Validation dataset size:", len(val_ds))
print("Test dataset size:", len(test_ds))

Train dataset size: 21733
Validation dataset size: 2859
Test dataset size: 97


In [ ]:
def collate(batch):
    out = {}
    for k in batch[0].keys():
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out


train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

---

## Некоторые функции для обучения моделей и отрисовки результатов

In [ ]:
def smape_torch(y, yhat, eps=1e-6):
    return (2.0 * (yhat - y).abs() / (y.abs() + yhat.abs() + eps)).mean()


def smape_numpy(y, yhat, eps=1e-6):
    return (2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)).mean()


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


@torch.no_grad()
def validation_loop(model, dl, return_preds=False, quantile_idx=1):
    model.eval()

    losses, smapes = [], []
    if return_preds:
        all_y_true = []
        all_y_pred = []
        all_store = []
        all_fam = []
        all_ctx = []

    for batch in dl:
        batch = to_device(batch, DEVICE)

        y = batch["y"]  # (B, H)
        out = model(batch)

        if out.ndim == 3 and out.shape[-1] == 3:
            yhat = out[:, :, quantile_idx]
        else:
            yhat = out  # (B, H)

        loss = F.mse_loss(yhat, y)
        sm = smape_torch(y, yhat)

        losses.append(loss.item())
        smapes.append(sm.item())

        if return_preds:
            all_y_true.append(y.detach().cpu())
            all_y_pred.append(yhat.detach().cpu())
            all_store.append(batch["store_id"].detach().cpu())
            all_fam.append(batch["family_id"].detach().cpu())
            all_ctx.append(batch["x_y"].detach().cpu())

    mean_loss = float(np.mean(losses))
    mean_sm = float(np.mean(smapes))

    if not return_preds:
        return mean_loss, mean_sm

    pack = {
        "y_true": torch.cat(all_y_true, dim=0).numpy(),  # (N, H)
        "y_pred": torch.cat(all_y_pred, dim=0).numpy(),  # (N, H)
        "store_id": torch.cat(all_store, dim=0).numpy(),  # (N, )
        "family_id": torch.cat(all_fam, dim=0).numpy(),  # (N, )
        "x_y": torch.cat(all_ctx, dim=0).numpy(),  # (N, L)
    }
    return mean_loss, mean_sm, pack


def train_loop(model, train_dl, val_dl, epochs=5, lr=1e-3, wd=1e-4, max_grad=1.0):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        for batch in train_dl:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)

            y = batch["y"]
            yhat = model(batch)
            loss = F.mse_loss(yhat, y)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad)
            opt.step()

        tr_loss, tr_sm = validation_loop(model, train_dl)
        va_loss, va_sm = validation_loop(model, val_dl)

        print(
            f"Epoch {ep:02d} | train loss {tr_loss:.4f} smape {tr_sm:.4f} | val loss {va_loss:.4f} smape {va_sm:.4f}"
        )

        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

In [ ]:
def build_stats_from_series_data(series_data):
    stats = {}
    for _, d in series_data.items():
        stats[(int(d["store_id"]), int(d["family_id"]))] = (float(d["y_mu"]), float(d["y_sd"]))
    return stats


def denorm_to_sales(y_norm, mu, sd):
    return np.expm1(y_norm * sd + mu)


def _select_point_pred(y_pred, mode="median", quantile_index=1):
    if y_pred.ndim == 2:
        return y_pred
    if y_pred.ndim == 3:
        if mode == "median":
            return y_pred[:, :, quantile_index]
        if mode == "mean":
            return y_pred.mean(axis=-1)


def denorm_forecasts(preds, series_data, point_mode="median", quantile_index=1):
    stats = build_stats_from_series_data(series_data)

    y_true = preds["y_true"]
    y_pred = _select_point_pred(preds["y_pred"], mode=point_mode, quantile_index=quantile_index)

    store_ids = preds["store_id"].astype(int)
    fam_ids = preds["family_id"].astype(int)

    N, H = y_true.shape
    y_true_sales = np.empty((N, H), dtype=np.float32)
    y_pred_sales = np.empty((N, H), dtype=np.float32)

    for i in range(N):
        mu, sd = stats[(store_ids[i], fam_ids[i])]
        y_true_sales[i] = denorm_to_sales(y_true[i], mu, sd)
        y_pred_sales[i] = denorm_to_sales(y_pred[i], mu, sd)

    return y_true_sales, y_pred_sales


def plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=60,
    seed=42,
    point_mode="median",
    quantile_index=1,
):
    stats = build_stats_from_series_data(series_data)

    y_true = preds["y_true"]
    y_pred = _select_point_pred(preds["y_pred"], mode=point_mode, quantile_index=quantile_index)
    x_y = preds["x_y"]

    store_ids = preds["store_id"].astype(int)
    fam_ids = preds["family_id"].astype(int)

    N = y_true.shape[0]
    H = y_true.shape[1]
    L = x_y.shape[1]
    show_ctx = int(min(context_points, L))

    rng = random.Random(seed)
    idxs = rng.sample(range(N), k=min(n_series, N))

    figs = []

    for idx in idxs:
        sid = store_ids[idx]
        fid = fam_ids[idx]
        mu, sd = stats[(sid, fid)]

        ctx_norm = x_y[idx, -show_ctx:]  # (show_ctx,)
        true_norm = y_true[idx]  # (H,)
        pred_norm = y_pred[idx]  # (H,)

        ctx_sales = denorm_to_sales(ctx_norm, mu, sd)
        true_sales = denorm_to_sales(true_norm, mu, sd)
        pred_sales = denorm_to_sales(pred_norm, mu, sd)

        eps = 1e-6
        sm = float(
            np.mean(
                2
                * np.abs(pred_sales - true_sales)
                / (np.abs(true_sales) + np.abs(pred_sales) + eps)
            )
        )

        x_ctx = np.arange(-show_ctx + 1, 1)
        x_fut = np.arange(1, H + 1)

        title = f"store_id={sid}, family_id={fid} | sMAPE={sm:.3f}"

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x_ctx, y=ctx_sales, mode="lines", name="context (sales)"))
        fig.add_trace(go.Scatter(x=x_fut, y=true_sales, mode="lines+markers", name="true (sales)"))
        fig.add_trace(go.Scatter(x=x_fut, y=pred_sales, mode="lines+markers", name="pred (sales)"))

        fig.add_vline(x=0, line_dash="dash")

        fig.update_layout(
            title=title,
            xaxis_title="relative time (0 = forecast start)",
            yaxis_title="sales",
            hovermode="x unified",
            template="plotly_white",
            width=950,
            height=380,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            margin=dict(l=60, r=20, t=60, b=50),
        )

        fig.show()
        figs.append(fig)

    return figs

## 1. DLinear — выделение тренда и остатка + линейные слои для каждой компоненты

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.k = k
        self.pool = nn.AvgPool1d(kernel_size=k, stride=1, padding=k // 2, count_include_pad=False)

    def forward(self, x):  # x: (B, L, 1)
        x = x.transpose(1, 2)  # (B ,1, L)
        ma = self.pool(x)  # (B, 1, L)
        ma = ma.transpose(1, 2)  # (B, L, 1)
        return ma


class DLinear(nn.Module):
    def __init__(self, L, H, ma_k=25):
        super().__init__()
        self.ma = MovingAvg(ma_k)
        self.lin_tr = nn.Linear(L, H)
        self.lin_se = nn.Linear(L, H)

    def forward(self, batch):
        x = batch["x_y"].unsqueeze(-1)  # (B, L, 1)
        trend = self.ma(x)  # (B, L, 1)
        seasonal = x - trend  # (B, L, 1)
        tr = self.lin_tr(trend.squeeze(-1))  # (B, H)
        se = self.lin_se(seasonal.squeeze(-1))  # (B, H)
        return tr + se

In [ ]:
m_dlin = DLinear(L, H, ma_k=25)
print(m_dlin)
summary(
    m_dlin,
    input_data={
        "batch": {
            "x_y": torch.zeros(2, L),
            "y": torch.zeros(2, H),
            "x_past_cov": torch.zeros(2, L, len(PAST_COV_COLS)),
            "x_known_past": torch.zeros(2, L, len(KNOWN_FUTURE_COLS)),
            "x_known_fut": torch.zeros(2, H, len(KNOWN_FUTURE_COLS)),
            "store_id": torch.zeros(2, dtype=torch.long),
            "family_id": torch.zeros(2, dtype=torch.long),
        }
    },
)

m_dlin = train_loop(m_dlin, train_dl, val_dl, epochs=10, lr=1e-3)
test_loss, test_sm, preds = validation_loop(m_dlin, test_dl, return_preds=True)
print("DLinear | test loss:", test_loss, "test sMAPE:", test_sm)

y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("DLinear | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

DLinear(
  (ma): MovingAvg(
    (pool): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(12,))
  )
  (lin_tr): Linear(in_features=90, out_features=14, bias=True)
  (lin_se): Linear(in_features=90, out_features=14, bias=True)
)
Epoch 01 | train loss 0.8640 smape 1.2197 | val loss 0.7339 smape 1.1637
Epoch 02 | train loss 0.8054 smape 1.1828 | val loss 0.6808 smape 1.1407
Epoch 03 | train loss 0.7894 smape 1.1651 | val loss 0.6629 smape 1.1185
Epoch 04 | train loss 0.7851 smape 1.1607 | val loss 0.6587 smape 1.1138
Epoch 05 | train loss 0.7837 smape 1.1594 | val loss 0.6614 smape 1.1125
Epoch 06 | train loss 0.7826 smape 1.1564 | val loss 0.6560 smape 1.1063
Epoch 07 | train loss 0.7827 smape 1.1577 | val loss 0.6584 smape 1.1126
Epoch 08 | train loss 0.7825 smape 1.1571 | val loss 0.6606 smape 1.1123
Epoch 09 | train loss 0.7826 smape 1.1556 | val loss 0.6588 smape 1.1098
Epoch 10 | train loss 0.7825 smape 1.1581 | val loss 0.6594 smape 1.1113
DLinear | test loss: 0.9026197195053101 t

## 2. CNN (TCNForecaster) + RNN (LSTNet)

In [ ]:
class Chomp1d(nn.Module):
    # Remove extra padded steps on the right to keep causality.
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, : -self.chomp_size] if self.chomp_size > 0 else x


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.0):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)  # causal padding handled via chomp
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.drop1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.drop2(self.relu2(self.chomp2(self.conv2(out))))
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TCNForecast(nn.Module):
    def __init__(self, H, channels=(32, 32, 32, 32), kernel_size=3, dropout=0.05):
        super().__init__()
        layers = []
        in_ch = 1
        for i, out_ch in enumerate(channels):
            dilation = 2**i
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, H)

    def forward(self, batch):
        # batch["x_y"]: (B, L)
        x = batch["x_y"].unsqueeze(1)  # (B, 1, L)
        y = self.tcn(x)  # (B, C, L)
        last = y[:, :, -1]  # (B, C)
        return self.head(last)  # (B, H)

In [ ]:
m_cnn = TCNForecast(H=H, channels=(32, 32, 32, 32), kernel_size=3, dropout=0.05)
print(m_cnn)
summary(
    m_cnn,
    input_data={
        "batch": {
            "x_y": torch.zeros(2, L),
            "y": torch.zeros(2, H),
            "x_past_cov": torch.zeros(2, L, len(PAST_COV_COLS)),
            "x_known_past": torch.zeros(2, L, len(KNOWN_FUTURE_COLS)),
            "x_known_fut": torch.zeros(2, H, len(KNOWN_FUTURE_COLS)),
            "store_id": torch.zeros(2, dtype=torch.long),
            "family_id": torch.zeros(2, dtype=torch.long),
        }
    },
)

m_cnn = train_loop(m_cnn, train_dl, val_dl, epochs=10, lr=1e-3)
test_loss, test_sm, preds = validation_loop(m_cnn, test_dl, return_preds=True)
print("TCN | test loss:", test_loss, "test sMAPE:", test_sm)
y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("TCN | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

TCNForecast(
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv1): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp1): Chomp1d()
      (relu1): ReLU()
      (drop1): Dropout(p=0.05, inplace=False)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (chomp2): Chomp1d()
      (relu2): ReLU()
      (drop2): Dropout(p=0.05, inplace=False)
      (downsample): Conv1d(1, 32, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
    )
    (1): TemporalBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (chomp1): Chomp1d()
      (relu1): ReLU()
      (drop1): Dropout(p=0.05, inplace=False)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (chomp2): Chomp1d()
      (relu2): ReLU()
      (drop2): Dropout(p=0.05, inplace=False)
      (relu): ReLU()
    )
    (2): TemporalBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(

In [ ]:
class LSTNetCore(nn.Module):
    def __init__(
        self,
        L,
        H,
        cnn_channels=64,
        cnn_kernel=6,
        rnn_hidden=64,
        ar_window=24,
        use_ar=True,
    ):
        super().__init__()
        self.L = L
        self.H = H
        self.use_ar = use_ar
        self.ar_window = ar_window
        self.cnn_kernel = cnn_kernel

        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=cnn_kernel),
            nn.ReLU(),
        )
        self.gru = nn.GRU(input_size=cnn_channels, hidden_size=rnn_hidden, batch_first=True)
        self.head = nn.Linear(rnn_hidden, H)

        if use_ar:
            self.ar = nn.Linear(ar_window, H)

    def forward(self, batch):
        # batch["x_y"]: (B, L)
        x = batch["x_y"].unsqueeze(-1)  # (B, L, 1)

        z = self.cnn(x.transpose(1, 2))  # (B, C, T') where T' = L - cnn_kernel + 1
        z = z.transpose(1, 2)  # (B, T', C)
        _, h = self.gru(z)  # h: (1, B, Hid)
        out = self.head(h[-1])  # (B, H)

        if self.use_ar:
            ar_in = x[:, -self.ar_window :, 0]  # (B, ar_window)
            out = out + self.ar(ar_in)  # (B, H)

        return out

In [ ]:
m_lstn = LSTNetCore(
    L=L, H=H, cnn_channels=64, cnn_kernel=6, rnn_hidden=64, ar_window=24, use_ar=True
)
print(m_lstn)
summary(
    m_lstn,
    input_data={
        "batch": {
            "x_y": torch.zeros(2, L),
            "y": torch.zeros(2, H),
            "x_past_cov": torch.zeros(2, L, len(PAST_COV_COLS)),
            "x_known_past": torch.zeros(2, L, len(KNOWN_FUTURE_COLS)),
            "x_known_fut": torch.zeros(2, H, len(KNOWN_FUTURE_COLS)),
            "store_id": torch.zeros(2, dtype=torch.long),
            "family_id": torch.zeros(2, dtype=torch.long),
        }
    },
)

m_lstn = train_loop(m_lstn, train_dl, val_dl, epochs=10, lr=1e-3)
test_loss, test_sm, preds = validation_loop(m_lstn, test_dl, return_preds=True)
print("LSTNet | test loss:", test_loss, "test sMAPE:", test_sm)
y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("LSTNet | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

LSTNetCore(
  (cnn): Sequential(
    (0): Conv1d(1, 64, kernel_size=(6,), stride=(1,))
    (1): ReLU()
  )
  (gru): GRU(64, 64, batch_first=True)
  (head): Linear(in_features=64, out_features=14, bias=True)
  (ar): Linear(in_features=24, out_features=14, bias=True)
)
Epoch 01 | train loss 0.9005 smape 1.2191 | val loss 0.7334 smape 1.1465
Epoch 02 | train loss 0.8181 smape 1.1687 | val loss 0.6615 smape 1.0909
Epoch 03 | train loss 0.7881 smape 1.1538 | val loss 0.6394 smape 1.0787
Epoch 04 | train loss 0.7711 smape 1.1458 | val loss 0.6265 smape 1.0718
Epoch 05 | train loss 0.7651 smape 1.1333 | val loss 0.6315 smape 1.0647
Epoch 06 | train loss 0.7630 smape 1.1171 | val loss 0.6162 smape 1.0409
Epoch 07 | train loss 0.7495 smape 1.1004 | val loss 0.6088 smape 1.0194
Epoch 08 | train loss 0.7410 smape 1.1177 | val loss 0.6216 smape 1.0540
Epoch 09 | train loss 0.7324 smape 1.1000 | val loss 0.6181 smape 1.0415
Epoch 10 | train loss 0.7213 smape 1.0783 | val loss 0.6131 smape 1.0097
LS

## 4. Интервальное прогнозирование с экзогенными переменными в TFT (Supervised Transformer Model)

In [ ]:
def quantile_loss(y, yhat, qs=(0.1, 0.5, 0.9)):
    # y: (B,H), yhat: (B,H,3)
    y = y.unsqueeze(-1)  # (B,H,1)
    q = torch.tensor(qs, device=y.device).view(1, 1, -1)
    e = y - yhat
    return torch.maximum(q * e, (q - 1) * e).mean()


@torch.no_grad()
def evaluate_tft(model, dl):
    model.eval()
    losses, smapes = [], []
    for batch in dl:
        for k in batch:
            batch[k] = batch[k].to(DEVICE)
        y = batch["y"]  # (B,H)
        qhat = model(batch)  # (B,H,3)
        loss = quantile_loss(y, qhat)
        yhat = qhat[:, :, 1]
        losses.append(loss.item())
        smapes.append(smape_torch(y, yhat).item())
    return float(np.mean(losses)), float(np.mean(smapes))


def train_tft(model, train_dl, val_dl, epochs=5, lr=1e-3, wd=1e-4, max_grad=1.0):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        for batch in train_dl:
            for k in batch:
                batch[k] = batch[k].to(DEVICE)

            y = batch["y"]
            qhat = model(batch)
            loss = quantile_loss(y, qhat)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad)
            opt.step()

        tr_loss, tr_sm = evaluate_tft(model, train_dl)
        va_loss, va_sm = evaluate_tft(model, val_dl)
        print(
            f"Epoch {ep:02d} | train qloss {tr_loss:.4f} smape(med) {tr_sm:.4f} | val qloss {va_loss:.4f} smape(med) {va_sm:.4f}"
        )

        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

In [ ]:
class MiniTFT(nn.Module):
    def __init__(
        self,
        L,
        H,
        n_stores,
        n_fams,
        d_static=16,
        d_model=64,
        past_cov_dim=2,
        known_dim=len(KNOWN_FUTURE_COLS),
        nhead=4,
    ):
        super().__init__()
        self.L, self.H = L, H

        self.emb_store = nn.Embedding(n_stores, d_static)
        self.emb_fam = nn.Embedding(n_fams, d_static)
        self.static_proj = nn.Linear(2 * d_static, d_model)

        # входы encoder: [y, past_cov, known_past]
        self.enc_in = nn.Linear(1 + past_cov_dim + known_dim, d_model)
        # входы decoder: [known_future]
        self.dec_in = nn.Linear(known_dim, d_model)

        self.enc_lstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model, num_layers=1, batch_first=True
        )
        self.dec_lstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model, num_layers=1, batch_first=True
        )

        # attention: decoder queries к encoder keys/values
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, batch_first=True)

        # прогнозируем 3 квантили (0.1, 0.5, 0.9)
        self.head = nn.Linear(d_model, 3)

    def forward(self, batch):
        B = batch["x_y"].size(0)

        st = self.emb_store(batch["store_id"])  # (B, d_static)
        fm = self.emb_fam(batch["family_id"])  # (B, d_static)
        static = self.static_proj(torch.cat([st, fm], dim=-1))  # (B, d_model)

        # encoder input
        x_y = batch["x_y"].unsqueeze(-1)  # (B, L, 1)
        x_pc = batch["x_past_cov"]  # (B, L, Cp)
        x_kp = batch["x_known_past"]  # (B, L, Cf)
        enc_x = torch.cat([x_y, x_pc, x_kp], dim=-1)  # (B, L, 1 + Cp + Cf)
        enc_x = self.enc_in(enc_x) + static.unsqueeze(1)  # (B, L, d_model)

        enc_out, (h, c) = self.enc_lstm(enc_x)  # (B,L,d_model)

        # decoder input (known future)
        dec_x = self.dec_in(batch["x_known_fut"]) + static.unsqueeze(1)  # (B, H, d_model)
        dec_out, _ = self.dec_lstm(dec_x, (h, c))  # (B, H, d_model)

        # attention: queries=dec_out, keys/values=enc_out
        attn_out, _ = self.attn(query=dec_out, key=enc_out, value=enc_out)  # (B, H, d_model)

        q = self.head(attn_out)  # (B, H, 3)
        return q

In [ ]:
stores = df["store_id"].unique().tolist()
items = df["item_id"].unique().tolist()

In [ ]:
m_tft = MiniTFT(L, H, n_stores=len(stores), n_fams=len(items), d_model=64)
print(m_tft)
summary(
    m_tft,
    input_data={
        "batch": {
            "x_y": torch.zeros(2, L),
            "y": torch.zeros(2, H),
            "x_past_cov": torch.zeros(2, L, len(PAST_COV_COLS)),
            "x_known_past": torch.zeros(2, L, len(KNOWN_FUTURE_COLS)),
            "x_known_fut": torch.zeros(2, H, len(KNOWN_FUTURE_COLS)),
            "store_id": torch.zeros(2, dtype=torch.long),
            "family_id": torch.zeros(2, dtype=torch.long),
        }
    },
)
m_tft = train_tft(m_tft, train_dl, val_dl, epochs=15, lr=1e-4)
test_loss, test_sm, preds = validation_loop(m_tft, test_dl, return_preds=True)
print("MiniTFT | test loss:", test_loss, "test sMAPE:", test_sm)
y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("MiniTFT | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

MiniTFT(
  (emb_store): Embedding(29, 16)
  (emb_fam): Embedding(34, 16)
  (static_proj): Linear(in_features=32, out_features=64, bias=True)
  (enc_in): Linear(in_features=10, out_features=64, bias=True)
  (dec_in): Linear(in_features=7, out_features=64, bias=True)
  (enc_lstm): LSTM(64, 64, batch_first=True)
  (dec_lstm): LSTM(64, 64, batch_first=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (head): Linear(in_features=64, out_features=3, bias=True)
)
Epoch 01 | train qloss 0.2386 smape(med) 1.7668 | val qloss 0.2364 smape(med) 1.7874
Epoch 02 | train qloss 0.2380 smape(med) 1.7999 | val qloss 0.2357 smape(med) 1.8222
Epoch 03 | train qloss 0.2372 smape(med) 1.7625 | val qloss 0.2348 smape(med) 1.7766
Epoch 04 | train qloss 0.2347 smape(med) 1.6694 | val qloss 0.2321 smape(med) 1.6722
Epoch 05 | train qloss 0.2166 smape(med) 1.2657 | val qloss 0.2104 smape(med) 1.2628
Epoch 06 | train qloss 0.2093 s

In [ ]:
test_loss, test_sm, preds = validation_loop(m_tft, test_dl, return_preds=True, quantile_idx=2)
print("MiniTFT | test loss:", test_loss, "test sMAPE:", test_sm)
y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("MiniTFT | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

MiniTFT | test loss: 1.6300960779190063 test sMAPE: 1.3939869403839111
MiniTFT | test sMAPE: after inverse transform: 0.44115132


In [ ]:
test_loss, test_sm, preds = validation_loop(m_tft, test_dl, return_preds=True, quantile_idx=0)
print("MiniTFT | test loss:", test_loss, "test sMAPE:", test_sm)
y_true_denorm, y_pred_denorm = denorm_forecasts(preds, series_data)
print("MiniTFT | test sMAPE: after inverse transform:", smape_numpy(y_true_denorm, y_pred_denorm))

figs = plot_results_plotly(
    preds,
    series_data,
    n_series=5,
    context_points=80,
    seed=42,
)

MiniTFT | test loss: 1.3879013061523438 test sMAPE: 1.249212622642517
MiniTFT | test sMAPE: after inverse transform: 0.44703218
